# HoloMine — Property Price Prediction From Sales Description (Task 2)

**Goal.** Predict `listPrice` (US dollars) from a single free-text property listing description.
**Metric.** Mean Absolute Error (MAE) on the **raw dollar scale** — lower is better.

---

## The one thing that decides this competition

The target is violently right-skewed: median \$499,900, mean \$839,073, max \$80,000,000.
If you predict the best possible constant (the median) you score **MAE ≈ 550,252**.

But look at *where the error lives*. Against that constant predictor:

| price band | rows | share of rows | share of total absolute error |
|---|---|---|---|
| < \$1M | 13,795 | 94.2% | 32% |
| \$1M – \$2M | 1,525 | 10.4% | 17% |
| \$2M – \$5M | 627 | 4.3% | 20% |
| > \$5M | 217 | 1.5% | 31% |

**5% of the rows carry ~half the metric.** Every design decision below follows from that:
we work in log space (so the model can reach \$20M at all), we pick losses that stay
well-conditioned in the tail, and we calibrate the final predictions to stretch them
back out toward the extremes.

## Strategy

1. **Log-target everywhere.** `log1p(y)` → predict → `expm1`. Training on raw dollars scores
   516,771 vs 346,843 for the identical model on the log target. Not close.
2. **Location is the signal.** Real estate is location. The text names towns, boroughs and
   landmarks; we extract them and build a *leak-free out-of-fold target encoding*. That single
   feature block has Spearman **0.522** with log price and is worth **~30,000 MAE** on its own.
3. **Diverse base models, blended in log space.** Sparse linear, kNN over TF-IDF cosine,
   GBDTs on engineered features, an ordinal threshold ensemble, and a fine-tuned DeBERTa-v3.
4. **Blend weights fit directly against OOF MAE**, then a **monotone affine calibration**
   in log space — worth another ~6,400 MAE because shrunk predictions systematically
   under-reach on expensive homes.
5. **Near-duplicate override.** ~1.5% of test listings are near-copies of a train listing.
   Where cosine ≥ 0.9, the neighbour's price is a far better estimate than any model.

Every number quoted in this notebook is a measured 5-fold out-of-fold result on a single
fixed split, so the comparisons are apples-to-apples.

In [ ]:
import os, re, gc, sys, json, time, math, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

# ------------------------------------------------------------------ config
class CFG:
    data_dir      = 'data'
    out_csv       = 'submission.csv'
    seed          = 42
    n_folds       = 5
    n_strat_bins  = 20        # quantile bins of log-price used to stratify the folds

    # floor/ceiling applied to every prediction (train range is $1 .. $80M)
    clip_lo       = 1_000.0
    clip_hi       = 9e7

    # transformer stage — set run_transformer=False for a CPU-only run
    run_transformer = False   # set True to fine-tune from scratch (~75 min on a 12 GB GPU)
    # deberta-v3-large scores better (OOF 283,793 vs 291,596) but costs 3.5 h vs 1.2 h on a
    # 12 GB card. Swap the block below to reproduce the stronger run:
    #   nn_model='microsoft/deberta-v3-large', nn_bs=4, nn_accum=4, nn_eval_bs=16,
    #   nn_epochs=3, nn_lr=8e-6, nn_head_lr=3e-5, nn_reinit=1, nn_gradckpt=True
    nn_model      = 'microsoft/deberta-v3-base'
    nn_max_len    = 512       # covers 98.6% of listings without truncation
    nn_bs         = 8         # 7.6 GB peak; bs=16 needs 12.6 GB and is no faster (bandwidth-bound)
    nn_accum      = 2         # -> effective batch 16
    nn_eval_bs    = 32
    nn_epochs     = 4
    nn_lr         = 2e-5
    nn_head_lr    = 1e-4
    nn_reinit     = 0         # re-initialise the last N encoder layers (1 helps for large)
    nn_gradckpt   = False     # required to fit deberta-v3-large in 12 GB
    nn_llrd       = 0.9       # layerwise LR decay
    nn_evals_per_epoch = 2    # keep the best checkpoint, not the last

    n_threads     = 8

os.makedirs('artifacts', exist_ok=True)
np.random.seed(CFG.seed)
print('numpy', np.__version__, '| pandas', pd.__version__)

In [ ]:
from sklearn.model_selection import StratifiedKFold

train = pd.read_csv(os.path.join(CFG.data_dir, 'train.csv'))
test  = pd.read_csv(os.path.join(CFG.data_dir, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(CFG.data_dir, 'sample_submission.csv'))

y      = train['listPrice'].values.astype(np.float64)
log_y  = np.log1p(y)
txt_tr = train['text'].astype(str).tolist()
txt_te = test['text'].astype(str).tolist()
N_TR, N_TE = len(train), len(test)

print(f'train {train.shape} | test {test.shape} | sample_sub {sample_sub.shape}')
print(f'nulls: train={train.isna().sum().sum()} test={test.isna().sum().sum()}')
print(f'duplicate ids: {train.id.duplicated().sum() + test.id.duplicated().sum()}')
print(f'exact text overlap train<->test: {len(set(txt_tr) & set(txt_te))}')
print(f'sample_submission id order == test id order: {(sample_sub.id.values == test.id.values).all()}')

### Target distribution and the error-concentration argument

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(b, float))))

print(train.listPrice.describe(percentiles=[.001, .01, .05, .25, .5, .75, .95, .99, .999]))
print()
const_med = float(np.median(y))
print(f'best constant (median {const_med:,.0f})  -> MAE {mae(y, np.full_like(y, const_med)):>12,.0f}')
print(f'constant mean   ({y.mean():,.0f})  -> MAE {mae(y, np.full_like(y, y.mean())):>12,.0f}')

bands = [(0, 2e5), (2e5, 5e5), (5e5, 1e6), (1e6, 2e6), (2e6, 5e6), (5e6, 1e12)]
tot = np.abs(y - const_med).sum()
rows = []
for lo, hi in bands:
    m = (y >= lo) & (y < hi)
    rows.append({'band': f'{lo:,.0f}-{hi:,.0f}', 'rows': int(m.sum()),
                 'pct_rows': 100 * m.mean(),
                 'pct_of_total_error': 100 * np.abs(y[m] - const_med).sum() / tot})
print()
print(pd.DataFrame(rows).to_string(index=False))

### Fold assignment

Plain `KFold` on a target this skewed produces folds whose tails differ wildly, which makes
OOF MAE noisy and makes model comparison unreliable. We stratify on 20 quantile bins of the
log price instead. **Every model in this notebook uses this exact same split**, so every OOF
number is directly comparable.

In [ ]:
def make_folds(log_target, n_splits, seed, n_bins):
    bins = pd.qcut(log_target, n_bins, labels=False, duplicates='drop')
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    f = np.zeros(len(log_target), dtype=int)
    for k, (_, vi) in enumerate(skf.split(log_target, bins)):
        f[vi] = k
    return f

folds = make_folds(log_y, CFG.n_folds, CFG.seed, CFG.n_strat_bins)
print('fold sizes:', np.bincount(folds))
for k in range(CFG.n_folds):
    m = folds == k
    print(f'  fold {k}: n={m.sum():5d}  median={np.median(y[m]):>10,.0f}  '
          f'mean={y[m].mean():>11,.0f}  max={y[m].max():>12,.0f}')

### Scoring helper

`evaluate` prints the headline OOF MAE plus the per-band breakdown, because a model that
improves the overall number while getting *worse* above \$2M is usually a step backwards.

In [ ]:
REGISTRY = {}   # name -> dict(oof=..., test=..., mae=...)

def evaluate(name, oof_pred, test_pred, register=True, verbose=True):
    oof_pred  = np.clip(np.nan_to_num(oof_pred,  nan=const_med), CFG.clip_lo, CFG.clip_hi)
    test_pred = np.clip(np.nan_to_num(test_pred, nan=const_med), CFG.clip_lo, CFG.clip_hi)
    m = mae(y, oof_pred)
    if verbose:
        print(f'\n=== {name} ===\nOOF MAE = {m:,.1f}')
        tot_err = np.abs(y - oof_pred).sum()
        for lo, hi in bands:
            msk = (y >= lo) & (y < hi)
            if msk.sum() == 0:
                continue
            e = np.abs(y[msk] - oof_pred[msk])
            print(f'   {lo:>12,.0f}-{hi:<14,.0f} n={msk.sum():5d}  '
                  f'bandMAE={e.mean():>12,.0f}  {100*e.sum()/tot_err:5.1f}% of error')
    if register:
        REGISTRY[name] = {'oof': oof_pred, 'test': test_pred, 'mae': m}
    return m

## 1. Handcrafted features

Listing prose encodes structured facts in free form: *"4 bed, 2.5 bath, 2,400 sq ft on 0.3
acres, built 1998"*. Regexes recover them where present (bedrooms 52%, baths 41%, sqft 18%,
acres 14% coverage) and `NaN` elsewhere — LightGBM handles missingness natively, so we do
**not** impute.

On top of that we count keyword families. The strongest single handcrafted signals measured
against log-price are `baths` (ρ=0.481 where present), `bed_bath` (0.413), `lux_total`
(0.393) and, notably, **raw text length** (`n_char`, ρ=0.373) — expensive listings simply
get more words written about them.

In [ ]:
NUMW = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7,
        'eight': 8, 'nine': 9, 'ten': 10, 'a': 1, 'an': 1, 'single': 1, 'double': 2, 'triple': 3}
NUMWORD = r'(?:\d+(?:\.\d+)?|one|two|three|four|five|six|seven|eight|nine|ten)'
SQFT_PAT = (r'([\d,]{3,7})\s*(?:\+/-\s*)?'
            r'(?:sq\.?\s*(?:ft|feet)|square (?:feet|foot|footage)|sf\b|sqft)')
ACRE_PAT = r'(\d+(?:\.\d+)?)\s*(?:\+/-\s*)?(?:acres?\b|ac\b)'

KW = {
 'lux_estate': [r'\bestate\b', r'\bgated\b', r'\bprivate gate', r'\bgrand\b', r'\bmagnificent\b',
                r'\bexquisite\b', r'\bmajestic\b', r'\bopulent\b', r'\bpalatial\b',
                r'\bprestigious\b', r'\btrophy\b', r'\bcompound\b', r'\bluxur'],
 'lux_finish': [r'\bmarble\b', r'\bquartz\b', r'\bgranite\b', r'\bgourmet kitchen\b',
                r'\bchefs? kitchen\b', r'\bsub[- ]?zero\b', r'\bwolf\b', r'\bviking\b',
                r'\bthermador\b', r'\bmiele\b', r'\bgaggenau\b', r'\bwaterworks\b',
                r'\bcustom cabinet', r'\bhardwood floor', r'\bcrown moldin', r'\bcoffered\b',
                r'\bvaulted\b', r'\bcathedral ceilin', r'\bfloor[- ]to[- ]ceiling\b',
                r'\bbespoke\b', r'\bartisan\b', r'\bradiant\b'],
 'lux_amenity': [r'\bwine cellar\b', r'\belevator\b', r'\bguest house\b', r'\bcasita\b',
                 r'\bhome theater\b', r'\bmedia room\b', r'\btheater room\b', r'\bwine room\b',
                 r'\bsauna\b', r'\bsteam room\b', r'\btennis court\b', r'\bpickleball\b',
                 r'\bputting green\b', r'\bhelipad\b', r'\bbowling\b',
                 r'\bgolf (?:course|simulator)\b', r'\bstaff quarters\b', r'\bau pair\b',
                 r'\bbutler\b', r'\bcarriage house\b', r'\bpool house\b', r'\bpenthouse\b'],
 'pool': [r'\bpool\b', r'\bspa\b', r'\bhot tub\b', r'\bjacuzzi\b', r'\binfinity edge\b'],
 'water': [r'\bwaterfront\b', r'\blakefront\b', r'\boceanfront\b', r'\bbeachfront\b',
           r'\briverfront\b', r'\bbayfront\b', r'\bwater ?view', r'\bdock\b', r'\bboat slip\b',
           r'\bmarina\b', r'\bseawall\b', r'\bcanal\b', r'\bfrontage on\b', r'\bon the water\b',
           r'\bbeach access\b', r'\bshoreline\b'],
 'view': [r'\bviews?\b', r'\bpanoramic\b', r'\bvista\b', r'\bskyline\b', r'\bmountain view',
          r'\bsunset\b', r'\bbreathtaking\b'],
 'new_constr': [r'\bnew construction\b', r'\bnewly (?:built|constructed)\b', r'\bto be built\b',
                r'\bbrand new\b', r'\bnever lived in\b', r'\bbuilder\b',
                r'\bpre[- ]?construction\b', r'\bmove[- ]?in ready\b'],
 'renovated': [r'\brenovat', r'\bremodel', r'\bupdated\b', r'\bupgrade', r'\brefinish',
               r'\brestored\b', r'\bnew roof\b', r'\bnew hvac\b', r'\bnew furnace\b',
               r'\bnew windows\b', r'\bturn[- ]?key\b'],
 'distress': [r'\bfixer\b', r'\bfixer[- ]upper\b', r'\btlc\b', r'\bas[- ]is\b',
              r'\bneeds work\b', r'\bhandyman\b', r'\bforeclosure\b', r'\bshort sale\b',
              r'\bauction\b', r'\bbank owned\b', r'\breo\b', r'\bcash only\b', r'\binvestor\b',
              r'\bsweat equity\b', r'\bteardown\b', r'\bcondemned\b', r'\bsalvage\b',
              r'\bdeferred maintenance\b', r'\bpotential\b', r'\bopportunity\b', r'\brehab\b'],
 'mobile': [r'\bmobile home\b', r'\bmanufactured home\b', r'\bdouble ?wide\b',
            r'\bsingle ?wide\b', r'\btrailer\b', r'\bpark model\b', r'\bmodular\b'],
 'land': [r'\bvacant land\b', r'\bland only\b', r'\bbuildable lot\b', r'\braw land\b',
          r'\bundeveloped\b', r'\bparcel\b', r'\bacreage\b', r'\bhomesite\b',
          r'\blot for sale\b', r'\btimber\b', r'\bpasture\b', r'\bperc test\b', r'\bzoned\b',
          r'\boff grid\b', r'\brecreational\b'],
 'condo': [r'\bcondo\b', r'\bcondominium\b', r'\bco[- ]?op\b', r'\bunit \d', r'\bhoa\b',
           r'\bassociation fee', r'\bcommon charge'],
 'townhouse': [r'\btownhouse\b', r'\btownhome\b', r'\brow house\b', r'\bend[- ]unit\b'],
 'multifam': [r'\bmulti[- ]?family\b', r'\bduplex\b', r'\btriplex\b', r'\bfourplex\b',
              r'\bquadplex\b', r'\b\d+[- ]unit\b', r'\btwo[- ]family\b', r'\bthree[- ]family\b',
              r'\brental income\b', r'\bcap rate\b', r'\bnoi\b', r'\btenant',
              r'\bincome producing\b', r'\bgross income\b'],
 'commercial': [r'\bcommercial\b', r'\bretail\b', r'\bwarehouse\b', r'\boffice space\b',
                r'\bindustrial\b', r'\bstorefront\b', r'\brestaurant\b', r'\bmixed[- ]use\b'],
 'farm': [r'\bfarm\b', r'\branch\b', r'\bbarn\b', r'\bequestrian\b', r'\bstable', r'\bpaddock\b',
          r'\bcorral\b', r'\bvineyard\b', r'\borchard\b', r'\bcattle\b', r'\bsilo\b', r'\bhay\b',
          r'\birrigat'],
 'garage': [r'\bgarage\b', r'\bcarport\b', r'\bparking\b', r'\bport[- ]?cochere\b',
            r'\bdriveway\b'],
 'basement': [r'\bbasement\b', r'\bcellar\b', r'\blower level\b', r'\bwalk[- ]?out\b'],
 'fireplace': [r'\bfireplace\b', r'\bwood stove\b', r'\bhearth\b'],
 'smart': [r'\bsmart home\b', r'\bnest\b', r'\bsolar\b', r'\bgenerator\b', r'\bev charg',
           r'\btesla\b', r'\bgeothermal\b', r'\bheat pump\b', r'\bmini[- ]split\b', r'\bsonos\b'],
 'hist': [r'\bhistoric\b', r'\bvictorian\b', r'\bcolonial\b', r'\bcraftsman\b', r'\btudor\b',
          r'\bfarmhouse\b', r'\bbungalow\b', r'\bcape\b', r'\branch[- ]style\b',
          r'\bmid[- ]century\b', r'\bcontemporary\b', r'\bmediterranean\b', r'\bspanish\b',
          r'\bgeorgian\b', r'\bfederal\b', r'\bbrownstone\b', r'\bpre[- ]?war\b'],
 'urban': [r'\bdowntown\b', r'\bwalk(?:able|ing distance)\b', r'\bsubway\b', r'\bmetro\b',
           r'\btransit\b', r'\bcommut', r'\btrain station\b', r'\bbus line\b',
           r'\bcity center\b', r'\bhighway\b', r'\bfreeway\b', r'\binterstate\b', r'\bdoorman\b'],
 'school': [r'\bschools?\b', r'\bdistrict\b', r'\buniversity\b', r'\bcollege\b'],
 'privacy': [r'\bprivate\b', r'\bsecluded\b', r'\bcul[- ]?de[- ]?sac\b', r'\bwooded\b',
             r'\bprivacy\b', r'\bserene\b', r'\btranquil\b'],
 'starter': [r'\bstarter home\b', r'\bfirst[- ]time\b', r'\baffordable\b', r'\bcozy\b',
             r'\bcharming\b', r'\bquaint\b', r'\bcompact\b', r'\bvalue\b', r'\bbudget\b'],
 'season': [r'\bcabin\b', r'\bcottage\b', r'\bchalet\b', r'\bretreat\b', r'\bvacation\b',
            r'\bsecond home\b', r'\bshort[- ]term rental\b', r'\bairbnb\b', r'\bski\b'],
 'hoa_fee': [r'\bhoa\b', r'\bmonthly fee', r'\bmaintenance fee', r'\bassessment'],
 'age55': [r'\b55\+', r'\bage[- ]restricted\b', r'\bactive adult\b', r'\bsenior\b'],
}
KW_C = {k: [re.compile(p, re.I) for p in v] for k, v in KW.items()}


def _num(s):
    if s is None:
        return np.nan
    s = str(s).strip().lower().replace(',', '')
    if s in NUMW:
        return float(NUMW[s])
    try:
        return float(s)
    except Exception:
        return np.nan


def _first(pat, t):
    m = re.search(pat, t, re.I)
    return _num(m.group(1)) if m else np.nan


def _all(pat, t):
    return [_num(m.group(1)) for m in re.finditer(pat, t, re.I)]


def extract_row(t):
    t = str(t); tl = t.lower(); f = {}

    beds = _first(r'(' + NUMWORD + r')\s*(?:\+\s*)?[- ]?\s*(?:bed(?:room)?s?|br\b|bdrm|bd\b)', t)
    if np.isnan(beds):
        beds = _first(r'bed(?:room)?s?\s*[:\-]?\s*(\d+)', t)
    f['beds'] = beds
    baths = _first(r'(' + NUMWORD + r')\s*(?:\+\s*)?[- ]?\s*(?:full\s+)?(?:bath(?:room)?s?|ba\b)', t)
    if np.isnan(baths):
        baths = _first(r'bath(?:room)?s?\s*[:\-]?\s*(\d+(?:\.\d+)?)', t)
    f['baths'] = baths
    f['half_bath'] = 1.0 if re.search(r'half[- ]bath|powder room', tl) else 0.0

    sq = [v for v in _all(SQFT_PAT, t) if v == v and 100 <= v <= 60000]
    f['sqft'] = max(sq) if sq else np.nan
    f['sqft_n'] = float(len(sq))
    ac = [v for v in _all(ACRE_PAT, t) if v == v and 0 < v <= 100000]
    f['acres'] = max(ac) if ac else np.nan
    yrs = [v for v in _all(r'\b(1[6-9]\d{2}|20[0-4]\d)\b', t) if v == v and 1600 <= v <= 2030]
    f['year_built'] = min(yrs) if yrs else np.nan
    f['year_max'] = max(yrs) if yrs else np.nan
    f['age'] = (2025 - f['year_built']) if f['year_built'] == f['year_built'] else np.nan
    f['garage_cars'] = _first(r'(' + NUMWORD + r')\s*[- ]?car\b', t)
    f['stories'] = _first(r'(' + NUMWORD + r')[- ]stor(?:y|ies)\b', t)
    f['units'] = _first(r'(' + NUMWORD + r')[- ]unit\b', t)

    dl = [v for v in _all(r'\$\s?([\d,]+(?:\.\d+)?)', t) if v == v and v > 0]
    f['dollar_n'] = float(len(dl))
    f['dollar_max_log'] = np.log1p(max(dl)) if dl else np.nan
    f['dollar_med_log'] = np.log1p(float(np.median(dl))) if dl else np.nan

    words = t.split()
    f['n_char'] = float(len(t)); f['n_word'] = float(len(words))
    f['n_sent'] = float(t.count('.') + t.count('!') + t.count('?'))
    f['avg_wlen'] = float(np.mean([len(w) for w in words])) if words else 0.0
    f['n_excl'] = float(t.count('!'))
    nd = sum(c.isdigit() for c in t)
    f['n_digit'] = float(nd); f['digit_ratio'] = nd / max(1, len(t))
    up = [w for w in words if len(w) > 2 and w.isupper()]
    f['caps_ratio'] = len(up) / max(1, len(words))
    f['n_comma'] = float(t.count(','))
    f['uniq_ratio'] = len(set(w.lower() for w in words)) / max(1, len(words))
    f['redacted'] = float(t.count('[Redacted'))

    for k, pats in KW_C.items():
        c = sum(len(p.findall(t)) for p in pats)
        f['kw_' + k] = float(c)
        f['kwb_' + k] = 1.0 if c > 0 else 0.0

    ok = lambda v: v == v
    f['sqft_per_bed'] = (f['sqft'] / f['beds']) if (ok(f['sqft']) and ok(f['beds']) and f['beds'] > 0) else np.nan
    f['bed_bath'] = (f['beds'] * f['baths']) if (ok(f['beds']) and ok(f['baths'])) else np.nan
    f['log_sqft'] = np.log1p(f['sqft']) if ok(f['sqft']) else np.nan
    f['log_acres'] = np.log1p(f['acres']) if ok(f['acres']) else np.nan
    f['lux_total'] = f['kw_lux_estate'] + f['kw_lux_finish'] + f['kw_lux_amenity']
    f['lux_density'] = f['lux_total'] / max(1.0, f['n_word'] / 100.0)
    return f


t0 = time.time()
HAND_TR = pd.DataFrame([extract_row(t) for t in txt_tr]).astype('float32')
HAND_TE = pd.DataFrame([extract_row(t) for t in txt_te]).astype('float32')
HAND_COLS = list(HAND_TR.columns)
print(f'handcrafted features: {HAND_TR.shape} in {time.time()-t0:.0f}s')

from scipy.stats import spearmanr
rows = []
for c in HAND_COLS:
    v = HAND_TR[c].values.astype(float)
    m = ~np.isnan(v)
    if m.sum() < 200 or np.nanstd(v) == 0:
        continue
    rows.append((c, float(spearmanr(v[m], log_y[m]).statistic), float(m.mean())))
rows.sort(key=lambda z: -abs(z[1]))
print('\ntop 15 handcrafted features by |Spearman| vs log-price (on rows where present):')
for c, r, cov in rows[:15]:
    print(f'   {c:<18s} rho={r:+.3f}   coverage={cov:.2f}')

## 2. Location target encoding — the single biggest feature

Nothing prices a house like where it is. The listings name places directly
("in Portland", "near Chelsea", "Klamath Falls, OR"), so we mine candidate place strings with
three patterns: after a preposition (`in|near|of|at …`), a `City, ST` pattern, and full state
names. We also keep generic Title-Case tokens as weak features.

Measured on this data: the vocabulary has **35,981** candidate tokens (6,727 seen ≥3 times),
and **95.7% of test listings contain at least one token the training set knows**. Median
price by token ranges from Klamath Falls (\$87,000) to the Hamptons (\$2,495,000) — that
spread *is* the signal.

**Leakage control.** The encoding is fit fold-wise: predictions for fold *k* only ever use
token statistics computed from the other four folds. Test-time statistics use all of train.
Smoothing pulls rare tokens toward the global prior, and per-document aggregation weights
*rarer* tokens higher, since "Chelsea" is far more informative than "Oregon".

In [ ]:
STOP_TITLE = set("""The This That A An In On At Of For With From Your You Our We It Is Are Be And But Or If As By To
Located Situated Welcome Step Enter Nestled Beautiful Stunning Charming Spacious Lovely Gorgeous Perfect Great New
Home House Property Living Kitchen Bedroom Bedrooms Bathroom Bathrooms Master Primary Suite Family Room Dining
Enjoy Featuring Features Offering Offers Come Call Schedule Seller Buyer Agent Listing Price Square Feet Acre
Acres Yard Garage Basement Floor Floors First Second Third Main Upper Lower Large Open Bright Natural Light
Whether Once Inside Outside Upstairs Downstairs Additional Plus Also Both All Each Every There Here Now Today
Recent Recently Newly Fully Beautifully Conveniently Ideally Just Only Minutes Close Walk Easy Short Owner
Redacted Entity North South East West Central Old Historic Modern Custom Private Public Community Level
Monday Tuesday Wednesday Thursday Friday Saturday Sunday January February March April May June July August
September October November December Spring Summer Fall Winter Buyers Sellers Sqft Full Half Bath
Please Note Must Will Can May Make Take Find Look Show Bring Offer Ready Move Back Front Side Over Under
Around Between Along Through After Before While Since Until Because Where When What Which Who Why How
One Two Three Four Five Six Seven Eight Nine Ten Lot Land Road Street Drive Lane Court Avenue""".split())

STATES = set("""AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY
NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC""".split())
STATE_NAMES = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
               "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
               "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
               "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
               "New Hampshire", "New Jersey", "New Mexico", "New York", "North Carolina",
               "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island",
               "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
               "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
STATE_RE = re.compile(r'\b(' + '|'.join(STATE_NAMES) + r')\b')
CTX_LOC = re.compile(r'(?:in|of|to|near|at|from|outside|within|throughout|downtown|around|beyond)\s+'
                     r'([A-Z][a-zA-Z]{2,}(?:\s+[A-Z][a-zA-Z]{2,}){0,2})\b')
CITY_STATE = re.compile(r'\b([A-Z][a-zA-Z]{2,}(?:\s[A-Z][a-zA-Z]{2,}){0,2}),\s*([A-Z]{2})\b')
TITLE_SEQ = re.compile(r'\b([A-Z][a-zA-Z]{2,}(?:\s+[A-Z][a-zA-Z]{2,}){0,2})\b')


def _clean_cand(cand):
    parts = [p for p in cand.split() if p not in STOP_TITLE]
    return ' '.join(parts) if parts else None


def loc_tokens(t):
    t = str(t); out = set()
    for m in CTX_LOC.finditer(t):
        c = _clean_cand(m.group(1))
        if c:
            out.add(c)
            if ' ' in c:
                out.add(c.split()[0])
    for m in CITY_STATE.finditer(t):
        if m.group(2) in STATES:
            c = _clean_cand(m.group(1))
            if c:
                out.add(c)
            out.add('ST_' + m.group(2))
    for m in STATE_RE.finditer(t):
        out.add('ST_' + m.group(1).replace(' ', '_'))
    for m in TITLE_SEQ.finditer(t):
        c = m.group(1)
        if c in STOP_TITLE:
            continue
        c2 = _clean_cand(c)
        if c2 and len(c2) > 3:
            out.add('T_' + c2)
    return out


LOC_COLS = ['loc_te_w', 'loc_te_mean', 'loc_te_max', 'loc_te_min', 'loc_te_std',
            'loc_n_known', 'loc_n_cand', 'loc_logmaxn', 'loc_logminn', 'loc_te_rarest']


class LocTE:
    """Smoothed target encoder over a bag of location tokens per document."""

    def __init__(self, min_count=3, smooth=20.0, max_tokens=40):
        self.min_count, self.smooth, self.max_tokens = min_count, smooth, max_tokens

    def fit(self, tok_lists, target):
        target = np.asarray(target, float)
        self.prior = float(target.mean())
        s, c = defaultdict(float), defaultdict(int)
        for toks, val in zip(tok_lists, target):
            for tk in toks:
                s[tk] += val; c[tk] += 1
        self.stats = {}
        for tk, cnt in c.items():
            if cnt < self.min_count:
                continue
            te = (s[tk] + self.smooth * self.prior) / (cnt + self.smooth)
            self.stats[tk] = (te, cnt)
        return self

    def transform(self, tok_lists):
        P = self.prior
        out = np.zeros((len(tok_lists), len(LOC_COLS)), dtype=np.float32)
        for i, toks in enumerate(tok_lists):
            hits = [self.stats[tk] for tk in toks if tk in self.stats]
            if not hits:
                out[i] = [P, P, P, P, 0, 0, len(toks), 0, 0, P]
                continue
            v = np.array([h[0] for h in hits])
            cnt = np.array([h[1] for h in hits], float)
            keep = np.argsort(-cnt)[:self.max_tokens]
            v, cnt = v[keep], cnt[keep]
            w = 1.0 / np.log1p(cnt)                    # rarer token -> more specific -> more weight
            out[i] = [float(np.average(v, weights=w)), float(v.mean()), float(v.max()),
                      float(v.min()), float(v.std()) if len(v) > 1 else 0.0,
                      float(len(v)), float(len(toks)), float(np.log1p(cnt.max())),
                      float(np.log1p(cnt.min())), float(v[np.argmin(cnt)])]
        return out


t0 = time.time()
TOK_TR = [loc_tokens(t) for t in txt_tr]
TOK_TE = [loc_tokens(t) for t in txt_te]

from collections import Counter
cnt = Counter()
for s_ in TOK_TR:
    cnt.update(s_)
known = {k for k, v in cnt.items() if v >= 3}
print(f'location vocab: {len(cnt):,} candidates, {len(known):,} seen >=3 times  ({time.time()-t0:.0f}s)')
print(f'test rows containing >=1 known token: {np.mean([len(s_ & known) > 0 for s_ in TOK_TE]):.3f}')

# strictly out-of-fold encoding for train; full-train encoding for test
LOC_TR = np.zeros((N_TR, len(LOC_COLS)), dtype=np.float32)
for k in range(CFG.n_folds):
    m = folds != k
    enc = LocTE().fit([TOK_TR[i] for i in np.where(m)[0]], log_y[m])
    LOC_TR[folds == k] = enc.transform([TOK_TR[i] for i in np.where(~m)[0]])
LOC_TE_ = LocTE().fit(TOK_TR, log_y).transform(TOK_TE)
LOC_TR = pd.DataFrame(LOC_TR, columns=LOC_COLS)
LOC_TE_ = pd.DataFrame(LOC_TE_, columns=LOC_COLS)

print('\nSpearman of each location feature vs log-price:')
for c in LOC_COLS:
    print(f'   {c:<16s} {spearmanr(LOC_TR[c].values, log_y).statistic:+.3f}')

## 3. Text vectorisation

Word 1–2 grams catch vocabulary ("wine cellar", "manufactured home"); `char_wb` 3–5 grams
catch morphology, typos and the run-together words this corpus is full of
("amenitiesthis", "areaperfect"). The union beats either alone. A 256-component SVD of the
word matrix gives the GBDTs a dense view of the same information.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

ALL_TXT = txt_tr + txt_te
t0 = time.time()
vec_w = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=300_000, sublinear_tf=True,
                        strip_accents='unicode', token_pattern=r'(?u)\b\w+\b')
vec_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=300_000,
                        sublinear_tf=True, strip_accents='unicode')
TW = vec_w.fit_transform(ALL_TXT)
TC = vec_c.fit_transform(ALL_TXT)
TFIDF = sp.hstack([TW, TC]).tocsr()
X_SP, XT_SP = TFIDF[:N_TR], TFIDF[N_TR:]
print(f'tf-idf: word {TW.shape}, char {TC.shape}, union {TFIDF.shape}  ({time.time()-t0:.0f}s)')

t0 = time.time()
svd = TruncatedSVD(n_components=256, random_state=CFG.seed, n_iter=7)
SVD_ALL = svd.fit_transform(TW).astype(np.float32)
SVD_TR, SVD_TE = SVD_ALL[:N_TR], SVD_ALL[N_TR:]
print(f'svd 256: explained variance {svd.explained_variance_ratio_.sum():.3f}  ({time.time()-t0:.0f}s)')

TFIDF_N = normalize(TFIDF)
XN_SP, XNT_SP = TFIDF_N[:N_TR], TFIDF_N[N_TR:]

## 4. Base models

### 4.1 Sparse linear models

The bar to clear. `LinearSVR` with `epsilon=0` is an L1-type loss in log space; `Ridge` is L2.
Both land near 346k, which is already 37% better than the constant baseline.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR

def cv_sparse(name, make_model, X, Xt):
    oof = np.zeros(N_TR); tp = np.zeros(Xt.shape[0])
    for k in range(CFG.n_folds):
        m = folds != k
        mdl = make_model()
        mdl.fit(X[m], log_y[m])
        oof[~m] = mdl.predict(X[~m])
        tp += mdl.predict(Xt) / CFG.n_folds
    return evaluate(name, np.expm1(oof), np.expm1(tp))

cv_sparse('ridge_tfidf',  lambda: Ridge(alpha=1.0, solver='lsqr', max_iter=3000), X_SP, XT_SP)
cv_sparse('svr_tfidf',    lambda: LinearSVR(C=0.5, epsilon=0.0, max_iter=6000, tol=1e-4), X_SP, XT_SP)

### 4.2 Sparse linear + handcrafted + location

The linear model cannot see the location encoding through TF-IDF alone, so we append the
dense block, rank-normalised onto a comparable scale.

In [ ]:
from sklearn.preprocessing import QuantileTransformer

DENSE_ALL = np.vstack([
    np.hstack([HAND_TR.values, LOC_TR.values]),
    np.hstack([HAND_TE.values, LOC_TE_.values]),
])
DENSE_ALL = np.nan_to_num(DENSE_ALL, nan=0.0, posinf=0.0, neginf=0.0)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='uniform',
                         subsample=200_000, random_state=CFG.seed)
DENSE_Q = qt.fit_transform(DENSE_ALL).astype(np.float32)
DENSE_SP = sp.csr_matrix(DENSE_Q)

AUG = sp.hstack([TFIDF, DENSE_SP * 1.0]).tocsr()
cv_sparse('ridge_aug', lambda: Ridge(alpha=1.0, solver='lsqr', max_iter=3000),
          AUG[:N_TR], AUG[N_TR:])
cv_sparse('svr_aug',   lambda: LinearSVR(C=0.5, epsilon=0.0, max_iter=6000, tol=1e-4),
          AUG[:N_TR], AUG[N_TR:])

### 4.3 kNN over TF-IDF cosine

A memory-based model: find the most similar training listings and average their log prices,
weighted by `cosine³`. It scores on par with the linear models but makes *completely
different* mistakes, which is exactly what a blend wants — it ends up with ~30% of the final
weight despite not being the strongest member.

It also gives us the near-duplicate detector used at the very end.

In [ ]:
def knn_predict(Q, R, y_ref, ks=(5, 10, 20), power=3.0, block=512):
    out = {k: np.zeros(Q.shape[0]) for k in ks}
    top1_sim = np.zeros(Q.shape[0]); top1_val = np.zeros(Q.shape[0])
    kmax = max(ks)
    for s in range(0, Q.shape[0], block):
        e = min(s + block, Q.shape[0])
        S = (Q[s:e] @ R.T).toarray()
        idx = np.argpartition(-S, kmax, axis=1)[:, :kmax]
        sims = np.take_along_axis(S, idx, 1)
        order = np.argsort(-sims, axis=1)
        idx = np.take_along_axis(idx, order, 1)
        sims = np.take_along_axis(sims, order, 1)
        top1_sim[s:e] = sims[:, 0]; top1_val[s:e] = y_ref[idx[:, 0]]
        for k in ks:
            w = np.clip(sims[:, :k], 0, None) ** power
            w /= np.maximum(w.sum(1, keepdims=True), 1e-9)
            out[k][s:e] = (w * y_ref[idx[:, :k]]).sum(1)
    return out, top1_sim, top1_val


t0 = time.time()
KS = (5, 10, 20)
knn_oof = {k: np.zeros(N_TR) for k in KS}
OOF_TOP1_SIM = np.zeros(N_TR); OOF_TOP1_VAL = np.zeros(N_TR)
for k in range(CFG.n_folds):
    m = folds != k
    o, s1, v1 = knn_predict(XN_SP[~m], XN_SP[m], log_y[m], ks=KS)
    for kk in KS:
        knn_oof[kk][~m] = o[kk]
    OOF_TOP1_SIM[~m] = s1; OOF_TOP1_VAL[~m] = v1
knn_te, TE_TOP1_SIM, TE_TOP1_VAL = knn_predict(XNT_SP, XN_SP, log_y, ks=KS)
print(f'knn done ({time.time()-t0:.0f}s)')
for kk in KS:
    evaluate(f'knn{kk}', np.expm1(knn_oof[kk]), np.expm1(knn_te[kk]), verbose=(kk == 10))

### 4.4 Gradient boosting on handcrafted + location + SVD

**The loss choice matters more than the hyperparameters.** Measured on identical features and
folds:

| objective (on `log1p` target) | OOF MAE |
|---|---|
| **huber** | **330,534** |
| l2 | 332,156 |
| l1 | 345,727 |
| l1 on the **raw** dollar target | 354,946 |

The textbook argument says L1 gives the conditional median, and since the median commutes
with monotone transforms, `expm1(median of log y)` is exactly the MAE-optimal estimator.
In practice L1 loses by 15k. The reason is a gradient-budget problem rather than a flaw in
the theory: L1 gives every row a unit-magnitude gradient, so the 5.2% of listings above \$2M
receive 5.2% of the learning signal while carrying ~48% of the metric.

That also points at the fix. Writing `v = log1p(y)` and `u = log1p(y_hat)`, a first-order
expansion gives `|y - y_hat| ~= (1 + y) * |v - u|`, so raw-dollar MAE is approximately
**L1 on the log target weighted by the price itself**. Weighting by `y` outright collapses
the effective sample size, so we sweep `w = clip(y, 1, q99.5) ** P`, where `P = 0` recovers
the unweighted model exactly and makes this a clean A/B. Measured here:

| weight exponent P (L1 on log) | OOF MAE |
|---|---|
| 0.00 (unweighted) | 344,891 |
| **0.25** | **340,760** |
| 0.50 | 344,390 |
| 0.75 | 360,765 |
| 1.00 (`w = y`, the "exact" surrogate) | 386,999 |

This is the most interesting negative result in the notebook. The derivation is sound and it
predicts the right *direction* — mild weighting does help, and `P = 0.25` buys a real 4,131
MAE over unweighted L1. But the fully-weighted `w = y` version that the first-order algebra
says is *exactly* raw MAE comes in **42,108 worse** than not weighting at all.

The reason is variance, not bias. Weighting by price concentrates almost the entire gradient
budget on a few hundred luxury listings, so the effective sample size collapses and the model
ends up fitting noise in the tail while ignoring the other 94% of the data. A first-order
surrogate for the loss is not the same thing as a good estimator of it.

Plain Huber still beats every weighted L1 variant, because it addresses the same problem from
the other side: L2-like near zero (well-conditioned, so the trees actually fit) and L1-like in
the tail (robust to the \$1 and \$80M rows), without distorting the sample weights at all. We
keep huber and L2 as the GBDT members and let the weighted variants contribute through the
blend, where their different error structure is still worth something.

In [ ]:
import lightgbm as lgb

GBDT_TR = np.hstack([HAND_TR.values, LOC_TR.values, SVD_TR]).astype(np.float32)
GBDT_TE = np.hstack([HAND_TE.values, LOC_TE_.values, SVD_TE]).astype(np.float32)
GBDT_COLS = HAND_COLS + LOC_COLS + [f'svd{i}' for i in range(SVD_TR.shape[1])]
print('GBDT design matrix:', GBDT_TR.shape)


def cv_lgb(name, params, X=GBDT_TR, Xt=GBDT_TE, cols=None, rounds=4000):
    base = dict(objective='huber', learning_rate=0.03, num_leaves=63, min_data_in_leaf=20,
                feature_fraction=0.35, bagging_fraction=0.85, bagging_freq=1,
                lambda_l1=0.1, lambda_l2=1.0, verbosity=-1,
                num_threads=CFG.n_threads, seed=CFG.seed, force_col_wise=True)
    base.update(params)
    cols = cols or GBDT_COLS
    oof = np.zeros(N_TR); tp = np.zeros(Xt.shape[0]); booster = None
    for k in range(CFG.n_folds):
        m = folds != k
        ds = lgb.Dataset(X[m], log_y[m], feature_name=cols)
        dv = lgb.Dataset(X[~m], log_y[~m], feature_name=cols, reference=ds)
        booster = lgb.train(base, ds, rounds, valid_sets=[dv],
                            callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[~m] = booster.predict(X[~m], num_iteration=booster.best_iteration)
        tp += booster.predict(Xt, num_iteration=booster.best_iteration) / CFG.n_folds
    evaluate(name, np.expm1(oof), np.expm1(tp))
    return booster


bst = cv_lgb('lgb_huber', dict(objective='huber'))
cv_lgb('lgb_l2', dict(objective='l2'))

imp = pd.Series(bst.feature_importance('gain'), index=GBDT_COLS).sort_values(ascending=False)
print('\ntop 20 features by gain:')
print(imp.head(20).to_string())

### 4.5 Ordinal threshold ensemble

Instead of regressing a number, ask 29 yes/no questions: *is this listing worth more than
\$79,000? …more than \$250,000? …more than \$3.2M?* Each is a cheap binary logistic model on
TF-IDF, and the answers form a survival curve `P(price > t)`. Where that curve crosses 0.5 is
the **conditional median** — which is precisely the MAE-optimal point estimate, obtained by
construction rather than by hoping a regression loss finds it.

The individual questions are very learnable (AUC 0.91–0.96), and the thresholds are packed
more densely in the upper tail where the metric is decided.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

QS = np.concatenate([np.linspace(0.04, 0.90, 23),
                     np.array([0.93, 0.95, 0.965, 0.98, 0.99, 0.995])])
THR = np.quantile(log_y, QS)

t0 = time.time()
SURV_OOF = np.zeros((N_TR, len(THR))); SURV_TE = np.zeros((N_TE, len(THR)))
for b, thr in enumerate(THR):
    lab = (log_y > thr).astype(int)
    for k in range(CFG.n_folds):
        m = folds != k
        clf = LogisticRegression(C=2.0, max_iter=1500, solver='liblinear')
        clf.fit(X_SP[m], lab[m])
        SURV_OOF[~m, b] = clf.predict_proba(X_SP[~m])[:, 1]
        SURV_TE[:, b] += clf.predict_proba(XT_SP)[:, 1] / CFG.n_folds
print(f'ordinal ensemble: {len(THR)} thresholds x {CFG.n_folds} folds in {time.time()-t0:.0f}s')


def survival_to_median(S, thr):
    S = np.minimum.accumulate(S, axis=1)        # enforce monotonicity
    lo_ext = thr[0] - (thr[1] - thr[0]); hi_ext = thr[-1] + (thr[-1] - thr[-2])
    out = np.empty(S.shape[0])
    for i in range(S.shape[0]):
        s = S[i]
        if s[0] < 0.5:
            out[i] = lo_ext + (0.5 / max(1e-6, 1.0 - s[0])) * (thr[0] - lo_ext); continue
        if s[-1] > 0.5:
            out[i] = thr[-1] + ((s[-1] - 0.5) / max(1e-6, s[-1])) * (hi_ext - thr[-1]); continue
        j = int(np.argmax(s < 0.5))
        s0, s1 = s[j - 1], s[j]
        f = 0.5 if s0 <= s1 else (s0 - 0.5) / (s0 - s1)
        out[i] = thr[j - 1] + f * (thr[j] - thr[j - 1])
    return out


def survival_to_mean(S, thr):
    S = np.minimum.accumulate(S, axis=1)
    lo = thr[0] - (thr[1] - thr[0])
    w = np.diff(np.concatenate([[lo], thr]))
    return lo + (S * w).sum(1)


evaluate('ordinal_median', np.expm1(survival_to_median(SURV_OOF, THR)),
         np.expm1(survival_to_median(SURV_TE, THR)))
evaluate('ordinal_mean', np.expm1(survival_to_mean(SURV_OOF, THR)),
         np.expm1(survival_to_mean(SURV_TE, THR)), verbose=False)

### 4.6 DeBERTa-v3 fine-tune

The sparse models see words; a transformer sees *sentences*. "needs a little TLC but has good
bones" and "meticulously maintained" share vocabulary and mean opposite things for price.

Recipe notes, all of which matter here:

* **Target** = standardised `log1p(price)`; **loss** = L1 on that standardised target.
* **`max_len = 512`** covers 98.6% of listings (median is only 176 tokens, p99 is 542).
* **Length-grouped batching.** With median 176 tokens against a 512 cap, naive shuffling
  spends most of each batch on padding. Sorting by length inside shuffled mega-batches cuts
  wall-clock roughly in half at identical quality.
* **`batch_size = 8` with `accum = 2`.** Measured on a 12 GB card, DeBERTa-v3's disentangled
  attention peaks at 12.6 GB at bs=16/len=512 — over budget. bs=8 peaks at 7.6 GB with
  *identical* throughput (the step is memory-bandwidth-bound), so the smaller batch is
  strictly better here.
* **Layerwise LR decay (0.9)** — lower layers hold general language, upper layers do the
  task-specific work; they should not move at the same rate.
* **Dropout disabled in the backbone.** Standard for regression heads: dropout injects noise
  into a single scalar output and measurably hurts.
* **Evaluate twice per epoch, keep the best checkpoint by fold MAE**, never the last.

**base vs large, measured on the identical folds.** deberta-v3-large wins overall but not
uniformly, which is worth seeing rather than assuming:

| fold | large | base | delta |
|---|---|---|---|
| 0 | 279,859 | 295,508 | -15,649 |
| 1 | 270,025 | 274,679 | -4,654 |
| 2 | 273,319 | 270,378 | **+2,941** |
| 3 | 317,945 | 327,096 | -9,151 |
| 4 | 277,818 | 290,323 | -12,505 |
| **OOF** | **283,793** | 291,596 | **-7,803** |

Large converges more slowly (lr 8e-6 against 2e-5, plus a re-initialised final layer): at half
of epoch 1 it sat at 463,600 against base's 382,144, and only overtook after epoch 1. Judging
it on early-epoch numbers would have been a mistake. Note fold 2, where large actually loses --
a single fold is not enough evidence either way, which is what the 5-fold mean is for.

Set `CFG.run_transformer = False` to skip this stage; everything downstream adapts.

In [ ]:
NN_OK = False
if CFG.run_transformer:
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import Dataset, DataLoader, Subset
        from transformers import (AutoTokenizer, AutoModel, AutoConfig,
                                  get_cosine_schedule_with_warmup)
        NN_OK = torch.cuda.is_available()
        if NN_OK:
            print('GPU:', torch.cuda.get_device_name(0),
                  f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB',
                  '| torch', torch.__version__)
        else:
            print('CUDA not available -> skipping the transformer stage')
    except Exception as e:
        print('transformer stack unavailable ->', type(e).__name__, e)
else:
    print('CFG.run_transformer = False -> skipping')

In [ ]:
if NN_OK:
    class TxtDS(Dataset):
        def __init__(self, texts, tok, max_len, target=None):
            self.t = list(texts); self.tok = tok; self.ml = max_len
            self.y = None if target is None else np.asarray(target, np.float32)

        def __len__(self):
            return len(self.t)

        def __getitem__(self, i):
            e = self.tok(self.t[i], truncation=True, max_length=self.ml, padding=False)
            d = {'input_ids': e['input_ids'], 'attention_mask': e['attention_mask']}
            if self.y is not None:
                d['y'] = float(self.y[i])
            return d

    class Collate:
        def __init__(self, tok):
            self.tok = tok

        def __call__(self, batch):
            has_y = 'y' in batch[0]
            ys = [b.pop('y') for b in batch] if has_y else None
            out = self.tok.pad(batch, padding=True, return_tensors='pt')
            if has_y:
                out['y'] = torch.tensor(ys, dtype=torch.float32)
            return out

    class LengthGroupedSampler(torch.utils.data.Sampler):
        """Shuffle, then sort by length inside mega-batches so batches pad to similar lengths."""

        def __init__(self, lengths, batch_size, mega=50, seed=0):
            self.l = np.asarray(lengths); self.bs = batch_size
            self.mega = mega * batch_size; self.epoch = 0; self.seed = seed

        def __len__(self):
            return len(self.l)

        def set_epoch(self, e):
            self.epoch = e

        def __iter__(self):
            g = np.random.default_rng(self.seed + self.epoch)
            idx = g.permutation(len(self.l))
            batches = []
            for i in range(0, len(idx), self.mega):
                c = idx[i:i + self.mega]
                c = c[np.argsort(self.l[c], kind='stable')]
                batches += [c[j:j + self.bs] for j in range(0, len(c), self.bs)]
            order = g.permutation(len(batches))
            return iter([int(i) for b in order for i in batches[b]])

    class MeanPool(nn.Module):
        def forward(self, h, mask):
            m = mask.unsqueeze(-1).to(h.dtype)
            return (h * m).sum(1) / m.sum(1).clamp(min=1e-6)

    class Regressor(nn.Module):
        def __init__(self, name):
            super().__init__()
            cfg = AutoConfig.from_pretrained(name)
            for attr in ('hidden_dropout_prob', 'attention_probs_dropout_prob',
                         'hidden_dropout', 'attention_dropout'):
                if hasattr(cfg, attr):
                    setattr(cfg, attr, 0.0)
            self.backbone = AutoModel.from_pretrained(name, config=cfg)
            if CFG.nn_gradckpt:
                self.backbone.gradient_checkpointing_enable()
            self.pool = MeanPool()
            self.head = nn.Linear(cfg.hidden_size, 1)
            nn.init.normal_(self.head.weight, std=0.01)
            nn.init.zeros_(self.head.bias)
            for layer in list(self.backbone.encoder.layer)[len(self.backbone.encoder.layer)
                                                           - CFG.nn_reinit:] if CFG.nn_reinit else []:
                for mod in layer.modules():
                    if isinstance(mod, nn.Linear):
                        mod.weight.data.normal_(mean=0.0, std=cfg.initializer_range)
                        if mod.bias is not None:
                            mod.bias.data.zero_()
                    elif isinstance(mod, nn.LayerNorm):
                        mod.bias.data.zero_(); mod.weight.data.fill_(1.0)

        def forward(self, input_ids, attention_mask, **kw):
            h = self.backbone(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
            return self.head(self.pool(h, attention_mask)).squeeze(-1)

    def build_optimizer(model, lr, head_lr, wd, llrd):
        no_decay = ('bias', 'LayerNorm.weight', 'layer_norm')
        named = dict(model.named_parameters()); groups = []; seen = set()

        def add(params, lr_):
            dec = [p for n, p in params if not any(nd in n for nd in no_decay)]
            nod = [p for n, p in params if any(nd in n for nd in no_decay)]
            if dec:
                groups.append({'params': dec, 'lr': lr_, 'weight_decay': wd})
            if nod:
                groups.append({'params': nod, 'lr': lr_, 'weight_decay': 0.0})

        layers = list(model.backbone.encoder.layer)
        n_layers = len(layers)
        for i in range(n_layers):
            ps = [(n, p) for n, p in named.items()
                  if n.startswith('backbone.encoder.layer.' + str(i) + '.')]
            seen.update(n for n, _ in ps)
            add(ps, lr * (llrd ** (n_layers - 1 - i)))
        emb = [(n, p) for n, p in named.items() if n.startswith('backbone.embeddings')]
        seen.update(n for n, _ in emb)
        add(emb, lr * (llrd ** n_layers))
        rest = [(n, p) for n, p in named.items() if n.startswith('backbone.') and n not in seen]
        seen.update(n for n, _ in rest)
        add(rest, lr)
        add([(n, p) for n, p in named.items() if not n.startswith('backbone.')], head_lr)
        return torch.optim.AdamW(groups, lr=lr, eps=1e-6)

    @torch.no_grad()
    def nn_predict(model, dl, amp_dtype):
        model.eval(); out = []
        for b in dl:
            ids = b['input_ids'].cuda(non_blocking=True)
            am = b['attention_mask'].cuda(non_blocking=True)
            with torch.autocast('cuda', dtype=amp_dtype):
                out.append(model(ids, am).float().cpu().numpy())
        return np.concatenate(out)

    print('transformer components defined')

In [ ]:
if NN_OK:
    MU, SD = float(log_y.mean()), float(log_y.std())
    y_std = (log_y - MU) / SD
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tok = AutoTokenizer.from_pretrained(CFG.nn_model)
    coll = Collate(tok)

    def tok_lens(texts):
        return [min(CFG.nn_max_len,
                    len(tok(t, truncation=True, max_length=CFG.nn_max_len)['input_ids']))
                for t in texts]

    LEN_TR, LEN_TE = tok_lens(txt_tr), tok_lens(txt_te)
    print('token lengths: median %.0f  p95 %.0f  p99 %.0f  |  coverage@%d = %.3f'
          % (np.median(LEN_TR), np.percentile(LEN_TR, 95), np.percentile(LEN_TR, 99),
             CFG.nn_max_len, np.mean(np.array(LEN_TR) < CFG.nn_max_len)))

    ds_te = TxtDS(txt_te, tok, CFG.nn_max_len)
    ord_te = np.argsort(LEN_TE, kind='stable'); inv_te = np.argsort(ord_te)
    dl_te = DataLoader(Subset(ds_te, ord_te.tolist()), batch_size=CFG.nn_eval_bs,
                       shuffle=False, collate_fn=coll, num_workers=0, pin_memory=True)

    nn_oof = np.zeros(N_TR); nn_test = np.zeros(N_TE)
    t_start = time.time()
    for k in range(CFG.n_folds):
        m = folds != k
        tr_idx, va_idx = np.where(m)[0], np.where(~m)[0]
        ds_tr = TxtDS([txt_tr[i] for i in tr_idx], tok, CFG.nn_max_len, y_std[tr_idx])
        ds_va = TxtDS([txt_tr[i] for i in va_idx], tok, CFG.nn_max_len)
        l_tr = [LEN_TR[i] for i in tr_idx]; l_va = [LEN_TR[i] for i in va_idx]
        samp = LengthGroupedSampler(l_tr, CFG.nn_bs, seed=CFG.seed)
        dl_tr = DataLoader(ds_tr, batch_size=CFG.nn_bs, sampler=samp, collate_fn=coll,
                           num_workers=0, pin_memory=True, drop_last=True)
        ord_va = np.argsort(l_va, kind='stable'); inv_va = np.argsort(ord_va)
        dl_va = DataLoader(Subset(ds_va, ord_va.tolist()), batch_size=CFG.nn_eval_bs,
                           shuffle=False, collate_fn=coll, num_workers=0, pin_memory=True)

        torch.manual_seed(CFG.seed + k)
        model = Regressor(CFG.nn_model).cuda()
        opt = build_optimizer(model, CFG.nn_lr, CFG.nn_head_lr, 0.01, CFG.nn_llrd)
        n_steps = (len(dl_tr) // CFG.nn_accum) * CFG.nn_epochs
        sch = get_cosine_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
        scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))
        lossf = nn.L1Loss()
        eval_every = max(1, len(dl_tr) // CFG.nn_evals_per_epoch)
        best = (1e18, None, None)

        for ep in range(CFG.nn_epochs):
            samp.set_epoch(ep); model.train(); run = 0.0
            for i, b in enumerate(dl_tr):
                ids = b['input_ids'].cuda(non_blocking=True)
                am = b['attention_mask'].cuda(non_blocking=True)
                yy = b['y'].cuda(non_blocking=True)
                with torch.autocast('cuda', dtype=amp_dtype):
                    loss = lossf(model(ids, am).float(), yy) / CFG.nn_accum
                scaler.scale(loss).backward()
                run += loss.item() * CFG.nn_accum
                if (i + 1) % CFG.nn_accum == 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                    opt.zero_grad(set_to_none=True); sch.step()
                if (i + 1) % eval_every == 0 or (i + 1) == len(dl_tr):
                    pv = nn_predict(model, dl_va, amp_dtype)[inv_va]
                    cur = mae(y[va_idx],
                              np.clip(np.expm1(pv * SD + MU), CFG.clip_lo, CFG.clip_hi))
                    print(f'  fold{k} ep{ep} {i+1}/{len(dl_tr)} loss={run/(i+1):.4f} '
                          f'valMAE={cur:,.0f} ({time.time()-t_start:.0f}s)')
                    if cur < best[0]:
                        best = (cur, pv.copy(), nn_predict(model, dl_te, amp_dtype)[inv_te])
                    model.train()

        nn_oof[va_idx] = np.expm1(best[1] * SD + MU)
        nn_test += np.expm1(best[2] * SD + MU) / CFG.n_folds
        print(f'fold {k} best MAE {best[0]:,.0f}')
        del model, opt
        gc.collect(); torch.cuda.empty_cache()

    evaluate('deberta_v3', nn_oof, nn_test)
    np.save('artifacts/nn_oof.npy', nn_oof)
    np.save('artifacts/nn_test.npy', nn_test)
elif os.path.exists('artifacts/nn_oof.npy'):
    evaluate('deberta_v3', np.load('artifacts/nn_oof.npy'), np.load('artifacts/nn_test.npy'))
    print('(loaded cached transformer predictions from a previous run)')

## 5. Level-2 stack

A LightGBM trained on **every base model's out-of-fold prediction** plus the handcrafted,
location and SVD blocks. Where a plain blend applies one fixed weight everywhere, the stacker
can learn *conditional* trust: lean on kNN when a near neighbour exists, lean on the
transformer for long descriptive prose, lean on location encoding when the town is known.

The prediction columns are already out-of-fold, and the stacker is cross-validated on the
same fold assignment, so no target information leaks across the split.

In [ ]:
STACK_NAMES = sorted(REGISTRY.keys())
P_OOF = np.column_stack([np.log1p(REGISTRY[n]['oof']) for n in STACK_NAMES]).astype(np.float32)
P_TE = np.column_stack([np.log1p(REGISTRY[n]['test']) for n in STACK_NAMES]).astype(np.float32)


def spread_feats(P):
    return np.column_stack([P.mean(1), P.std(1), P.max(1) - P.min(1), np.median(P, 1)])


KNN_OOF = np.column_stack([knn_oof[k] for k in KS]
                          + [OOF_TOP1_SIM, OOF_TOP1_VAL]).astype(np.float32)
KNN_TE = np.column_stack([knn_te[k] for k in KS]
                         + [TE_TOP1_SIM, TE_TOP1_VAL]).astype(np.float32)

STK_TR = np.hstack([GBDT_TR, P_OOF, KNN_OOF, spread_feats(P_OOF)]).astype(np.float32)
STK_TE = np.hstack([GBDT_TE, P_TE, KNN_TE, spread_feats(P_TE)]).astype(np.float32)
STK_COLS = (GBDT_COLS + ['P_' + n for n in STACK_NAMES]
            + ['knn' + str(k) for k in KS] + ['knn_top1sim', 'knn_top1val']
            + ['p_mean', 'p_std', 'p_range', 'p_median'])
print('stack design matrix:', STK_TR.shape)

cv_lgb('stack_lgb_huber', dict(objective='huber', learning_rate=0.02, feature_fraction=0.3),
       X=STK_TR, Xt=STK_TE, cols=STK_COLS, rounds=6000)
cv_lgb('stack_lgb_l2', dict(objective='l2', learning_rate=0.02, feature_fraction=0.3),
       X=STK_TR, Xt=STK_TE, cols=STK_COLS, rounds=6000)

## 6. Blending

Three decisions, each made against measured OOF MAE rather than intuition.

**Which space to average in.** Averaging raw dollars lets one model shouting "$8M" drag the
consensus away from four models saying "$600k"; averaging logs bounds that disagreement but
systematically biases the result downward (the geometric mean is always below the arithmetic
one, by roughly `1 + s^2/2` where `s` is the per-row spread of log predictions). Rather than
argue, we nest both inside a **power mean** and sweep the exponent: `p -> 0` is the geometric
mean (log-space blending), `p = 1` is the arithmetic mean, and everything between is
available. The sweep picks the winner on OOF.

**How to fit the weights.** For a linear blend in raw space, MAE is a sum of absolute values
of affine functions -- it is **convex in the weights**, so there are no spurious local minima
and no need for hundreds of random restarts. We therefore also fit a **median regression**
(`QuantileRegressor(quantile=0.5)`), which is the exact L1-optimal linear stacker, folded
into the same CV. Caruana greedy forward selection with replacement runs alongside as the
overfitting-resistant option. We keep whichever wins on OOF.

**Whether to use the mean or the median of models.** With only a handful of members the
weighted mean is preferred: MAE is convex, so by Jensen the mean ensemble is guaranteed never
worse than the average member, and the median has no analogous guarantee. With an even number
of members the median also silently averages a different middle *pair* on every row.

Finally we report the **standard error of the OOF MAE**. On a target this heavy-tailed it is
several thousand dollars, and any "improvement" smaller than about twice that is noise.

In [ ]:
from scipy.optimize import minimize
from sklearn.linear_model import QuantileRegressor


def pmean(P, w, p):
    """Power mean. p->0 is the geometric mean (== blending in log space); p==1 is arithmetic."""
    P = np.clip(P, 1.0, None)
    if abs(p) < 1e-6:
        return np.expm1(np.log1p(P) @ w)
    return (P ** p @ w) ** (1.0 / p)


def fit_weights(P, target, p, n_restarts=6, seed=0):
    m = P.shape[1]
    rng = np.random.default_rng(seed)
    starts = [np.ones(m) / m]
    s0 = np.zeros(m)
    s0[int(np.argmin([mae(target, P[:, j]) for j in range(m)]))] = 1.0
    starts.append(0.7 * s0 + 0.3 / m)
    starts += [rng.dirichlet(np.ones(m)) for _ in range(n_restarts)]

    def obj(w):
        a = np.abs(w)
        return 1e18 if a.sum() <= 0 else mae(target, pmean(P, a / a.sum(), p))

    best = (None, 1e18)
    for st in starts:
        r = minimize(obj, st, method='Nelder-Mead',
                     options=dict(maxiter=5000, fatol=1e-4, adaptive=True))
        if r.fun < best[1]:
            best = (np.abs(r.x) / np.abs(r.x).sum(), float(r.fun))
    return best


def greedy_forward(P_raw, target, rounds=80):
    """Caruana selection with replacement, in log space."""
    L = np.log1p(np.clip(P_raw, 1, None))
    n, m = P_raw.shape
    cur = np.zeros(n); count = np.zeros(m); best = (None, 1e18)
    for r in range(rounds):
        scores = [mae(target, np.expm1((cur * r + L[:, j]) / (r + 1))) for j in range(m)]
        j = int(np.argmin(scores))
        cur = (cur * r + L[:, j]) / (r + 1)
        count[j] += 1
        if scores[j] < best[1] - 1e-9:
            best = (count / count.sum(), scores[j])
    return best


names = sorted(REGISTRY.keys())
OOF_M = np.column_stack([REGISTRY[n]['oof'] for n in names])
TEST_M = np.column_stack([REGISTRY[n]['test'] for n in names])

print('single-model OOF MAE')
for n, v in sorted(((n, REGISTRY[n]['mae']) for n in names), key=lambda z: z[1]):
    print(f'   {v:>12,.0f}  {n}')

# drop members that are near-copies of a stronger member
C = np.corrcoef(np.log1p(np.clip(OOF_M, 1, None)).T)
keep = []
for j in np.argsort([REGISTRY[n]['mae'] for n in names]):
    if all(C[j, k] < 0.995 for k in keep):
        keep.append(j)
    else:
        print(f'   drop {names[j]} (corr > 0.995 with a stronger member)')
names_k = [names[j] for j in keep]
OOF_K, TEST_K = OOF_M[:, keep], TEST_M[:, keep]
print(f'kept {len(names_k)} of {len(names)} members')

print('\nsimple aggregations')
print(f'   mean, raw space   : {mae(y, OOF_K.mean(1)):>12,.1f}')
print(f'   mean, log space   : {mae(y, np.expm1(np.log1p(OOF_K).mean(1))):>12,.1f}')
print(f'   median of models  : {mae(y, np.median(OOF_K, 1)):>12,.1f}')

CANDIDATES = {}

print('\noptimised weights, per blend space')
for p in (0.0, 0.25, 0.5, 1.0):
    w, m_ = fit_weights(OOF_K, y, p)
    CANDIDATES[f'power_mean_p{p}'] = (m_, w, p)
    print(f'   p={p:<5}          : {m_:>12,.1f}')

w_g, m_g = greedy_forward(OOF_K, y)
CANDIDATES['greedy_log'] = (m_g, w_g, 0.0)
print(f'   greedy (log)      : {m_g:>12,.1f}')

# median regression -- the exact L1-optimal linear stacker, nested on the same folds
qr_oof = np.zeros(N_TR); qr_test = np.zeros(N_TE)
try:
    for k in range(CFG.n_folds):
        m = folds != k
        qr = QuantileRegressor(quantile=0.5, alpha=1e-4, solver='highs')
        qr.fit(OOF_K[m], y[m])
        qr_oof[~m] = qr.predict(OOF_K[~m])
        qr_test += qr.predict(TEST_K) / CFG.n_folds
    m_qr = mae(y, qr_oof)
    print(f'   median regression : {m_qr:>12,.1f}')
except Exception as e:
    m_qr = np.inf
    print('   median regression failed:', type(e).__name__)

best_key = min(CANDIDATES, key=lambda k: CANDIDATES[k][0])
if m_qr < CANDIDATES[best_key][0]:
    blend_oof, blend_test = qr_oof, qr_test
    print(f'\n-> chosen: median regression   OOF MAE {m_qr:,.1f}')
else:
    m_, W, p = CANDIDATES[best_key]
    blend_oof = pmean(OOF_K, W, p)
    blend_test = pmean(TEST_K, W, p)
    print(f'\n-> chosen: {best_key}   OOF MAE {m_:,.1f}')
    print('   weights:')
    for j in np.argsort(-W):
        if W[j] > 0.005:
            print(f'      {W[j]:.3f}  {names_k[j]}')

blend_oof = np.clip(blend_oof, CFG.clip_lo, CFG.clip_hi)
blend_test = np.clip(blend_test, CFG.clip_lo, CFG.clip_hi)
evaluate('blend', blend_oof, blend_test, register=False)

SE_MAE = float(np.std(np.abs(y - blend_oof), ddof=1) / np.sqrt(N_TR))
print(f'\nstandard error of this OOF MAE = {SE_MAE:,.0f}')
print(f'-> treat any difference smaller than about {2*SE_MAE:,.0f} as noise, not improvement')

## 7. Calibration

Every regularised model shrinks toward the mean, so predictions come out systematically flat:
not high enough on expensive homes, not low enough on cheap ones. Since the metric is raw
dollars and the error is concentrated in the tail, that flattening is expensive.

We fit a monotone affine map in log space, `log(p') = a + b*(log p - c) + c`, choosing
`(a, b)` to **minimise raw MAE directly**. A fitted `b > 1` stretches predictions back out.

The gain is measured **nested**: the calibrator is fit on four folds and scored on the fifth,
so the number is honest rather than the calibrator grading its own homework.

**Whether this helps depends entirely on what it is calibrating, and the nested check is what
tells you.** Measured on this data:

| blend being calibrated | uncalibrated | + affine | verdict |
|---|---|---|---|
| early greedy log-space blend | 311,809 | **305,390** | apply (-6,419) |
| final quantile-regression blend | 275,313 | 275,872 | **skip** (+561) |

The early blend was a plain weighted average, so it really was shrunk and the affine stretch
recovered 6,419 MAE. The final blend is a **median regression** fitted against raw dollars --
it already has an intercept and free coefficients fitted to this exact loss, so it arrives
calibrated and any further stretching just adds variance. Isotonic (+3,590) and
piecewise-linear (+62,897) are worse still: they have far more freedom, look excellent
in-sample, and fall apart out-of-fold.

So the cell below does not hard-code a decision. It scores every calibrator nested and applies
one **only if it actually wins there**. On the final pipeline it correctly applies nothing.

In [ ]:
from sklearn.isotonic import IsotonicRegression


def fit_affine(p, target):
    lp = np.log1p(np.clip(p, 1, None)); c = float(lp.mean())
    obj = lambda ab: mae(target, np.expm1(ab[0] + ab[1] * (lp - c) + c))
    best = (None, 1e18)
    for b0 in (0.9, 1.0, 1.05, 1.15, 1.3):
        r = minimize(obj, [0.0, b0], method='Nelder-Mead',
                     options=dict(maxiter=3000, fatol=1e-5))
        if r.fun < best[1]:
            best = (r.x, float(r.fun))
    return dict(kind='affine', a=float(best[0][0]), b=float(best[0][1]), c=c)


def fit_isotonic(p, target):
    lp = np.log1p(np.clip(p, 1, None))
    iso = IsotonicRegression(out_of_bounds='clip').fit(lp, np.log1p(np.clip(target, 1, None)))
    return dict(kind='iso', iso=iso)


def fit_piecewise(p, target, n_knots=12, min_per_bin=150):
    lp = np.log1p(np.clip(p, 1, None)); lt = np.log1p(np.clip(target, 1, None))
    edges = np.unique(np.quantile(lp, np.linspace(0, 1, n_knots + 1)))
    xs, ys = [], []
    for i in range(len(edges) - 1):
        m = (lp >= edges[i]) & (lp <= edges[i + 1])
        if m.sum() >= min_per_bin:
            xs.append(float(np.median(lp[m]))); ys.append(float(np.median(lt[m])))
    if len(xs) < 3:
        return dict(kind='affine', a=0.0, b=1.0, c=float(lp.mean()))
    xs = np.array(xs)
    ys = IsotonicRegression(out_of_bounds='clip').fit(xs, np.array(ys)).predict(xs)
    return dict(kind='pw', x=xs, yv=ys)


def apply_calib(p, cal):
    lp = np.log1p(np.clip(p, 1, None))
    if cal['kind'] == 'affine':
        return np.expm1(cal['a'] + cal['b'] * (lp - cal['c']) + cal['c'])
    if cal['kind'] == 'iso':
        return np.expm1(cal['iso'].predict(lp))
    return np.expm1(np.interp(lp, cal['x'], cal['yv']))


def nested_score(p, target, fitter):
    out = np.array(p, float).copy()
    for k in range(CFG.n_folds):
        m = folds != k
        out[~m] = apply_calib(p[~m], fitter(p[m], target[m]))
    return mae(target, out), out


base_mae = mae(y, blend_oof)
print(f'uncalibrated blend: {base_mae:,.1f}\n')
cands = {'affine': fit_affine,
         'isotonic': fit_isotonic,
         'piecewise12': lambda p, t: fit_piecewise(p, t, 12),
         'piecewise20': lambda p, t: fit_piecewise(p, t, 20)}
scored = {}
for nm, fn in cands.items():
    s, _ = nested_score(blend_oof, y, fn)
    scored[nm] = s
    print(f'   {nm:<12s} nested OOF MAE = {s:>12,.1f}   ({s - base_mae:+,.1f})')

best_cal = min(scored, key=scored.get)
if scored[best_cal] < base_mae - 1.0:
    cal = cands[best_cal](blend_oof, y)
    _, cal_oof = nested_score(blend_oof, y, cands[best_cal])
    cal_test = apply_calib(blend_test, cal)
    shown = {k: round(v, 4) for k, v in cal.items() if k in ('a', 'b', 'c')}
    print(f'\n-> applying "{best_cal}" {shown}')
else:
    cal_oof, cal_test = blend_oof, blend_test
    print('\n-> no calibration applied (no honest gain)')
evaluate('blend+calibrated', cal_oof, cal_test, register=False)

## 8. Near-duplicate override

About 1.5% of test listings are near-verbatim copies of a training listing — the same builder
posting adjacent lots, or a relisting with a word changed. Measured on OOF, when the top
cosine neighbour sits at similarity ≥ 0.9, **copying that neighbour's price scores MAE ≈
33,600** against a model error around 330,000 on those same rows.

The effect is sharply threshold-dependent — below roughly 0.9 it reverses and copying *hurts*,
because "similar wording" stops implying "same development". So we sweep both the threshold
and the neighbour weight on OOF and apply the override only if it wins there.

In [ ]:
def dup_sweep(pred, target, sim, nb_logprice,
              thresholds=(0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.99),
              weights=(1.0, 0.75, 0.5)):
    rows = [{'thr': np.nan, 'w': 0.0, 'n': 0, 'oof_mae': mae(target, pred)}]
    lp = np.log1p(np.clip(pred, 1, None))
    for thr in thresholds:
        m = sim >= thr
        if m.sum() < 5:
            continue
        for w in weights:
            p = pred.copy()
            p[m] = np.expm1(w * nb_logprice[m] + (1 - w) * lp[m])
            rows.append({'thr': thr, 'w': w, 'n': int(m.sum()),
                         'oof_mae': mae(target, p)})
    return pd.DataFrame(rows).sort_values('oof_mae').reset_index(drop=True)


sweep = dup_sweep(cal_oof, y, OOF_TOP1_SIM, OOF_TOP1_VAL)
print(sweep.head(10).to_string(index=False))

best = sweep.iloc[0]
final_oof, final_test = cal_oof.copy(), cal_test.copy()
if np.isfinite(best['thr']) and best['oof_mae'] < mae(y, cal_oof) - 200:
    thr, w = float(best['thr']), float(best['w'])
    for pred, sim, nb in ((final_oof, OOF_TOP1_SIM, OOF_TOP1_VAL),
                          (final_test, TE_TOP1_SIM, TE_TOP1_VAL)):
        m = sim >= thr
        lp = np.log1p(np.clip(pred, 1, None))
        pred[m] = np.expm1(w * nb[m] + (1 - w) * lp[m])
    print(f'\n-> override applied at similarity >= {thr} with neighbour weight {w} '
          f'({int((TE_TOP1_SIM >= thr).sum())} test rows affected)')
else:
    print('\n-> no override applied (no OOF gain)')

FINAL_OOF_MAE = evaluate('FINAL', final_oof, final_test, register=False)

## 9. Submission

Guardrails first. A submission that silently loses a row, reorders ids, or emits a NaN
scores zero regardless of how good the model is, so every one of these is asserted.

In [ ]:
pred = np.clip(np.nan_to_num(final_test, nan=const_med), CFG.clip_lo, CFG.clip_hi)
sub = pd.DataFrame({'id': sample_sub['id'].values, 'listPrice': pred})

checks = {
    'row count matches sample_submission': len(sub) == len(sample_sub),
    'columns are exactly [id, listPrice]': list(sub.columns) == ['id', 'listPrice'],
    'id order matches sample_submission': bool((sub.id.values == sample_sub.id.values).all()),
    'ids are unique': bool(sub.id.is_unique),
    'id set matches test set': set(sub.id) == set(test.id),
    'no NaN or inf': bool(np.isfinite(sub.listPrice).all()),
    'all predictions strictly positive': bool((sub.listPrice > 0).all()),
    'predictions inside train price range': bool(sub.listPrice.max() <= y.max()),
}
for k, v in checks.items():
    print(f'  [{"PASS" if v else "FAIL"}] {k}')
assert all(checks.values()), 'submission guardrails failed'

sub.to_csv(CFG.out_csv, index=False)
baseline = mae(y, np.full_like(y, const_med))
print(f'\nwrote {CFG.out_csv}  ({len(sub)} rows)')
print(f'FINAL OOF MAE = {FINAL_OOF_MAE:,.1f}')
print(f'constant-median baseline = {baseline:,.1f}   '
      f'-> improvement {100*(1-FINAL_OOF_MAE/baseline):.1f}%')

print('\ndistribution sanity check (prediction vs train target)')
print(pd.DataFrame({
    'prediction': sub.listPrice.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]),
    'train_y': pd.Series(y).describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]),
}).to_string())

print('\nfirst rows of submission.csv')
print(sub.head().to_string(index=False))

## What moved the needle

Final measured result, 5-fold out-of-fold on one fixed split:

| stage | OOF MAE | vs baseline |
|---|---|---|
| constant median (best possible constant) | 550,252 | -- |
| TF-IDF + LinearSVR, log target | 345,815 | -37.2% |
| LightGBM huber on handcrafted + location + SVD | 330,534 | -39.9% |
| DeBERTa-v3-**base**, 5-fold fine-tune | 291,596 | -47.0% |
| DeBERTa-v3-**large**, 5-fold fine-tune | 283,793 | -48.4% |
| level-2 stack over all 42 OOF predictions | 273,214 | -50.3% |
| weighted blend (median regression) | 267,845 | -51.3% |
| + near-duplicate override | **267,201** | **-51.4%** |

**Standard error of the OOF MAE = 9,359.** Any difference smaller than roughly 18,900 is
inside the noise floor, which matters for reading this table honestly: the jump from the
baseline is ~29 standard errors and unambiguous, the transformer's edge over the GBDT is
~4 standard errors and real, but the last two rows differ by less than one standard error
and should be treated as a tie rather than a ranking.

### What worked

1. **Log target.** Worth ~170,000 MAE against training on raw dollars (346,843 vs 516,771 for
   the identical Ridge). MAE is minimised by the conditional median, and the median commutes
   with monotone transforms, so `expm1(median of log y)` is the right estimator by
   construction. No smearing or `exp(sigma^2/2)` correction -- those target the *mean* and
   would be actively wrong here.
2. **Location target encoding.** Worth ~30,000 on its own (405,497 -> 375,267). Spearman 0.522
   with log price, the strongest single feature, and 95.7% of test listings contain a place
   token the training set has seen.
3. **The transformer.** 291,596 against 330,534 for the best GBDT. Fine-tuning reads meaning
   that bag-of-ngrams cannot: "needs TLC but has good bones" and "meticulously maintained"
   share vocabulary and mean opposite things.
4. **Stacking and blending.** 291,596 -> 275,313. Optimising the weights against the metric
   is what mattered: equal weights over the same 33 models scored only 316,455.
5. **Huber over L1 in log space.** Worth ~15,000, against the textbook expectation.

### What did not work, and was measured rather than assumed

* **Training on raw dollars** -- 516,771 vs 346,843. The single largest trap available.
* **`w = y` dollar weighting.** The first-order algebra says raw MAE is exactly L1 on the log
  target weighted by price. Measured, it scores 386,999 -- **42,108 worse than not weighting
  at all**. Only a mild `P = 0.25` helped (340,760), and even that lost to plain Huber. A
  first-order surrogate for a loss is not the same thing as a good estimator of it.
* **Calibrating the final blend** -- affine +561, isotonic +3,590, piecewise +62,897, all
  nested. The same affine step gained 6,419 on the earlier, weaker blend.
* **Copying near-duplicate neighbours below ~0.9 cosine** -- reverses and hurts. Above 0.9 it
  helps, but only by 692 MAE, which is well inside the noise floor and kept mainly because it
  is principled rather than because it is measurable.
* **A 64-class multinomial regression-as-classification head** -- far too slow for its value;
  the ordinal threshold ensemble reaches the same conditional median at a fraction of the cost.

### Where the remaining error lives

Listings above \$2M are 5.8% of the data and still carry **47%** of the error. That is the
structural ceiling of this task: at the top end, two listings with near-identical prose can be
a \$3M house and a \$20M house, and the text simply does not contain the information needed to
tell them apart. Further gains would most likely come from more transformer seeds and
architectures rather than from more feature engineering.